In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -q -o "/content/drive/MyDrive/PCGITA.zip" -d "/content/"


In [ ]:
import os

print(os.listdir("/content/PCGITA"))

In [ ]:
print(os.listdir("/content/PCGITA/Vowels"))

In [ ]:
!pip install -q transformers torchaudio scikit-learn matplotlib seaborn pandas umap-learn


In [ ]:

import os
import torch
import torchaudio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import Wav2Vec2Processor, Wav2Vec2Model

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.decomposition import PCA


In [ ]:
# STEP 3 — Load PC-GITA dataset

import os
import pandas as pd

DATASET_PATH = "/content/PCGITA/Vowels"

data = []

# -------------------------
# Healthy speakers
# -------------------------

control_path = os.path.join(DATASET_PATH, "Control")

for vowel in ["A","E","I","O","U"]:
    vowel_folder = os.path.join(control_path, vowel)

    for file in os.listdir(vowel_folder):
        if file.endswith(".wav"):
            data.append({
                "filename": os.path.join("Control", vowel, file),
                "label": 0
            })


# -------------------------
# Parkinson speakers
# -------------------------

pd_path = os.path.join(DATASET_PATH, "Patologicas")

for vowel in ["A","E","I","O","U"]:
    vowel_folder = os.path.join(pd_path, vowel)

    for file in os.listdir(vowel_folder):
        if file.endswith(".wav"):
            data.append({
                "filename": os.path.join("Patologicas", vowel, file),
                "label": 1
            })


labels_df = pd.DataFrame(data)

print("Total samples:", len(labels_df))
print("\nClass distribution:")
print(labels_df["label"].value_counts())

labels_df.head()

In [ ]:
# Class distribution
sns.countplot(x="label", data=labels_df)
plt.title("Class Distribution (0 = Healthy, 1 = Parkinson)")
plt.show()

# Audio duration distribution (sample)
durations = []
for file in labels_df["filename"][:50]:
    waveform, sr = torchaudio.load(os.path.join(DATASET_PATH, file))
    durations.append(waveform.shape[1] / sr)

plt.hist(durations, bins=20)
plt.title("Audio Duration Distribution (Sample)")
plt.xlabel("Seconds")
plt.ylabel("Count")
plt.show()


In [ ]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

model.eval()
for param in model.parameters():
    param.requires_grad = False


In [ ]:
# Pick one sample
sample_row = labels_df.iloc[0]
sample_path = os.path.join(DATASET_PATH, sample_row["filename"])

# Load raw audio
waveform, sr = torchaudio.load(sample_path)

if sr != 16000:
    resampler = torchaudio.transforms.Resample(sr, 16000)
    waveform = resampler(waveform)
    sr = 16000

print("RAW AUDIO SHAPE:", waveform.shape)

# Plot raw waveform
plt.figure(figsize=(12,3))
plt.plot(waveform.squeeze().numpy())
plt.title("Raw Audio Waveform (Before wav2vec2)")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.show()

# wav2vec2 forward pass
inputs = processor(
    waveform.squeeze().numpy(), # cannot squeeze
    sampling_rate=16000,
    return_tensors="pt",
    padding=True
)

with torch.no_grad():
    outputs = model(inputs.input_values)

hidden_states = outputs.last_hidden_state
print("FRAME-LEVEL EMBEDDINGS SHAPE (T × D):", hidden_states.shape)

# Temporal pooling
mean_pool = hidden_states.mean(dim=1)
max_pool = hidden_states.max(dim=1).values
final_embedding = torch.cat([mean_pool, max_pool], dim=1) # use anyone

print("FINAL EMBEDDING SHAPE (After pooling):", final_embedding.shape)

# Plot final embedding
plt.figure(figsize=(12,3))
plt.plot(final_embedding.squeeze().numpy())
plt.title("wav2vec2 Feature Vector (After Temporal Pooling)")
plt.xlabel("Feature Index")
plt.ylabel("Feature Value")
plt.show()


In [ ]:
# ==============================
# STEP — Visualize Wav2Vec2 Embeddings (CORRECTED)
# ==============================

import matplotlib.pyplot as plt

# Select one audio sample
sample_row = labels_df.iloc[0]
sample_path = os.path.join(DATASET_PATH, sample_row["filename"])

# Load audio
waveform, sr = torchaudio.load(sample_path)

# Resample if needed
if sr != 16000:
    resampler = torchaudio.transforms.Resample(sr, 16000)
    waveform = resampler(waveform)

print("RAW AUDIO SHAPE:", waveform.shape)

# Plot raw waveform
plt.figure(figsize=(12,3))
plt.plot(waveform.squeeze().numpy())
plt.title("Raw Audio Waveform")
plt.show()

# Pass through Wav2Vec2
inputs = processor(
    waveform.squeeze().numpy(),
    sampling_rate=16000,
    return_tensors="pt",
    padding=True
)

with torch.no_grad():
    outputs = model(inputs.input_values)

hidden_states = outputs.last_hidden_state.squeeze(0)   # (T, 768)

print("SEQUENCE SHAPE (T × 768):", hidden_states.shape)

# Plot ONE FEATURE over time
plt.figure(figsize=(12,3))
plt.plot(hidden_states[:, 0].numpy())   # first feature across time
plt.title("Feature evolution over time (No Pooling)")
plt.xlabel("Time Step")
plt.ylabel("Feature Value")
plt.show()

Feature extarctoion


In [ ]:
def extract_features_seq(audio_path):
    waveform, sr = torchaudio.load(audio_path)

    if sr != 16000:
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)

    inputs = processor(
        waveform.squeeze().numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(inputs.input_values)

    hidden = outputs.last_hidden_state.squeeze(0)  # (T, 768)

    return hidden

In [ ]:
X_seq = []
y = []

for _, row in labels_df.iterrows():
    path = os.path.join(DATASET_PATH, row["filename"])

    try:
        feat = extract_features_seq(path).cpu().numpy().astype(np.float16)  #  reduce RAM
        X_seq.append(feat)
        y.append(row["label"])
    except:
        print("Error")

y = np.array(y)

In [ ]:
np.save("X_seq.npy", np.array(X_seq, dtype=object))
np.save("y.npy", y)

In [ ]:
X_seq = np.load("X_seq.npy", allow_pickle=True)
y = np.load("y.npy")

In [ ]:
from sklearn.model_selection import train_test_split

X_train_seq, X_test_seq, y_train, y_test = train_test_split(
    X_seq, y, test_size=0.2, stratify=y
)

In [ ]:
from torch.nn.utils.rnn import pad_sequence
import torch

# =========================
# STEP 1: Padding (same as yours)
# =========================
X_train = pad_sequence(
    [torch.tensor(x, dtype=torch.float32) for x in X_train_seq],
    batch_first=True
)

X_test = pad_sequence(
    [torch.tensor(x, dtype=torch.float32) for x in X_test_seq],
    batch_first=True
)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# =========================
# STEP 2:  VERY IMPORTANT FIX (truncate sequence)
# =========================
MAX_LEN = 400   # reduce memory (can tune later)

X_train = X_train[:, :MAX_LEN, :]
X_test  = X_test[:, :MAX_LEN, :]

In [ ]:

del X_train_seq
del X_test_seq

import gc
gc.collect()

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X_train, y_train)

loader = DataLoader(
    dataset,
    batch_size=16,   #  DO NOT increase
    shuffle=True
)

In [ ]:
import torch.nn as nn
import torch

class LSTMFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(768, 128, num_layers=2, batch_first=True)
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        out, _ = self.lstm(x)          # (B, T, 128)
        last_out = out[:, -1, :]       # (B, 128)

        logits = self.fc(last_out)     # (B,1)
        prob = torch.sigmoid(logits)   # sigmoid output

        return last_out, prob          # 👈 IMPORTANT

In [ ]:
model = LSTMFeatureExtractor()

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    model.train()
    total_loss = 0

    for batch_X, batch_y in loader:   # 🔥 batch training

        features, outputs = model(batch_X)

        outputs = outputs.squeeze()
        loss = criterion(outputs, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss}")

In [ ]:
model.eval()

with torch.no_grad():
    train_features, _ = model(X_train)   # (N,128)
    test_features, _ = model(X_test)

train_features = train_features.numpy()
test_features = test_features.numpy()

y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

svm = SVC(kernel='rbf')   # you can try 'linear' also

svm.fit(train_features, y_train_np)

y_pred = svm.predict(test_features)

print("Final Accuracy (LSTM + SVM):", accuracy_score(y_test_np, y_pred))